# 04 · Resultados y evaluación

Fase CRISP-DM: **Evaluación**.

Compara todos los modelos con las mismas fotos. Primero se genera la evaluación:
```
python -m src.main evaluar          # val: se puede repetir
```
Y solo al final, una vez:
```
python -m src.main congelar
python -m src.main evaluar --test --confirmo
```

**Cómo leer las métricas**
- **F1 macro**: promedio del F1 de good, regular y bad. Así la clase con menos fotos (regular) pesa lo mismo que las demás.
- **Malas como buenas**: frutas malas que el modelo dejaría pasar como buenas. Es el error más caro para el negocio.
- **IC 95 %**: rango probable del F1 si se repitiera con otras fotos. Se calcula remuestreando **grupos** completos, porque las fotos de un grupo se parecen entre sí.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src" / "config.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

def read(name):
    """Lee un resultado de results/; si no existe, explica qué comando lo genera."""
    path = config.RESULTS / name
    if not path.is_file():
        raise FileNotFoundError(f"Falta {path}. Corre primero los pasos del README.")
    return json.loads(path.read_text(encoding="utf-8"))

config.FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
partition = "test" if (config.RESULTS / "evaluacion_test.json").is_file() else "val"
report = read(f"evaluacion_{partition}.json")
print(f"Partición: {partition} · {report['n']} recortes en {report['n_groups']} grupos")
pd.DataFrame([{"modelo": name, "F1 macro": round(m["f1_macro"], 3),
               "F1 Kaggle (2 clases)": round(m.get("f1_kaggle", float("nan")), 3),
               "F1 AgriFreshNET (3 clases)": round(m.get("f1_agrifreshnet", float("nan")), 3),
               "recall bad": round(m["recall_bad"], 3), "malas como buenas": m["bad_to_good"]}
              for name, m in report["models"].items()])

In [ ]:
lo, hi = report["models"]["ensemble"]["f1_macro_ci95"]
print(f"Ensamble RF + CNN: F1 {report['models']['ensemble']['f1_macro']:.3f}, IC 95 % [{lo:.3f}; {hi:.3f}]")
print(f"Fruta: accuracy {report['fruit']['accuracy']:.3f}, F1 macro {report['fruit']['f1_macro']:.3f}")

## Matrices de confusión

In [ ]:
import numpy as np

names = list(report["models"])
fig, axes = plt.subplots(1, len(names), figsize=(3.6 * len(names), 3.4))
for ax, name in zip(axes, names):
    m = report["models"][name]
    matrix = np.asarray(m["confusion_matrix"])
    ax.imshow(matrix / matrix.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3), m["labels"]); ax.set_yticks(range(3), m["labels"])
    for i in range(3):
        for j in range(3):
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=9)
    ax.set_title(f"{name}\nF1 {m['f1_macro']:.3f}", fontsize=9)
    ax.set_xlabel("predicho"); ax.set_ylabel("real")
fig.tight_layout()
fig.savefig(config.FIGURES / f"confusion_{partition}.png", dpi=150)

## Fruta: qué se confunde con qué

In [ ]:
fruit = report["fruit"]
pd.DataFrame(fruit["confusion_matrix"], index=fruit["labels"], columns=fruit["labels"])

## Prueba de fuga (opcional): `python -m src.main fuga`

Misma validación cruzada, cambiando solo cómo se agrupan las fotos de AgriFreshNET. Si el F1 baja al agrupar más grueso, con grupos finos la nota estaba inflada. La última columna debe dar 0 en "week": ninguna foto de val comparte semana con train.

In [ ]:
if (config.RESULTS / "prueba_fuga.json").is_file():
    leak = read("prueba_fuga.json")
    display(pd.DataFrame([{"agrupación": level, "grupos en train": item["train_groups"],
                           "F1 AgriFreshNET": round(item["cv"]["agrifreshnet_3cl"]["mean"], 3),
                           "F1 global": round(item["cv"]["overall"]["mean"], 3),
                           "val que comparte sesión con train":
                               item["crossing"]["val_agri_rows_sharing_session_with_train"]}
                          for level, item in leak["levels"].items()]))
else:
    print("Aún no se corrió la prueba de fuga.")

## Fotos de celular (etiquetadas a ciegas): `python -m src.main celular evaluar --confirmo`

La prueba más cercana al uso real: fotos tomadas por la clase con celulares, que el modelo nunca vio, con el estado etiquetado sin mirar la app.

In [ ]:
if (config.RESULTS / "evaluacion_celular.json").is_file():
    phone = read("evaluacion_celular.json")
    print(f"{phone['n']} fotos de {phone['sessions']} sesiones · "
          f"fruta: accuracy {phone['fruit']['accuracy']:.3f} {phone['fruit']['accuracy_ci95']}")
    display(pd.DataFrame([{"modelo": n, "F1 macro": round(q["f1_macro"], 3),
                           "IC 95 %": [round(v, 3) for v in q["f1_macro_ci95"]],
                           "malas como buenas": q["bad_to_good"]}
                          for n, q in phone["quality"].items()]))
else:
    print("Aún no se evaluaron las fotos de celular.")